<h3>실습 안내</h3><p>짧은 Document 세 건을 실제 OpenAI 임베딩으로 저장합니다. 검색 개수 <code>k</code>만 1에서 2로 바꿔 반환값을 비교합니다.</p>

In [1]:
# 임베딩과 메모리 벡터스토어에 필요한 패키지를 설치합니다.
# ==는 검증된 버전을 고정하고 -q는 설치 로그를 줄입니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3"

Note: you may need to restart the kernel to use updated packages.


<h3>1. API 키 준비</h3><p>현재 작업 폴더에서 <code>.env</code>를 찾아 실제 임베딩 호출을 준비합니다.</p>

In [2]:
# env_file은 find_dotenv()가 찾은 .env 경로 문자열입니다.
# load_dotenv()는 파일을 로드하고 os.getenv()는 OPENAI_API_KEY 존재를 확인합니다.
import os
from dotenv import find_dotenv, load_dotenv

env_file = find_dotenv(usecwd=True)
assert env_file, ".env를 찾지 못했습니다."
load_dotenv(env_file)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 없습니다."
print("API 키 준비 완료")

API 키 준비 완료


<h3>2. 라벨이 있는 문서 준비</h3><p>검색 결과를 확인할 수 있게 각 Document에 서로 다른 <code>doc_id</code>를 넣습니다.</p>

In [3]:
# raw_documents의 각 튜플은 (문서 ID, 본문) 두 값을 묶습니다.
# for로 튜플을 doc_id와 content로 나누어 Document를 하나씩 추가합니다.
from langchain_core.documents import Document

raw_documents = [
    ("D01", "미르시립도서관의 정기 휴관일은 매주 월요일입니다."),
    ("D02", "해솔식당의 점심 영업은 오전 11시에 시작합니다."),
    ("D03", "별빛회사의 급여일은 매월 25일입니다."),
]
documents = []
for doc_id, content in raw_documents:
    document = Document(page_content=content, metadata={"doc_id": doc_id})
    documents.append(document)
print("문서 수:", len(documents), "타입:", type(documents[0]).__name__)
for document in documents:
    print(document.metadata, document.page_content)

문서 수: 3 타입: Document
{'doc_id': 'D01'} 미르시립도서관의 정기 휴관일은 매주 월요일입니다.
{'doc_id': 'D02'} 해솔식당의 점심 영업은 오전 11시에 시작합니다.
{'doc_id': 'D03'} 별빛회사의 급여일은 매월 25일입니다.


<h3>3. 임베딩 모델과 벡터스토어 만들기</h3><p>문서를 실제로 임베딩해 메모리 벡터스토어에 추가합니다.</p>

In [4]:
# embedding은 문장을 숫자 벡터로 바꾸는 OpenAI 모델 객체입니다.
# add_documents()는 문서 벡터와 원본 Document를 store에 저장합니다.
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(model="text-embedding-3-small")
store = InMemoryVectorStore(embedding=embedding)
stored_ids = store.add_documents(documents)
print("store 타입:", type(store).__name__)
print("저장된 벡터 ID 수:", len(stored_ids))

store 타입: InMemoryVectorStore
저장된 벡터 ID 수: 3


<h3>4. 기본값 k=1로 검색</h3><p>점수가 포함된 검색 결과에서 Document와 점수를 따로 출력합니다.</p>

In [5]:
# similarity_search_with_score()는 (Document, 점수) 튜플의 리스트를 반환합니다.
# k=1은 가장 관련 높은 결과 한 건만 요청합니다.
query = "도서관이 쉬는 요일은?"
hits_k1 = store.similarity_search_with_score(query, k=1)
print("반환 타입:", type(hits_k1).__name__, "개수:", len(hits_k1))
for document, score in hits_k1:
    print("doc_id:", document.metadata["doc_id"])
    print("score:", round(score, 3))
    print("content:", document.page_content)

반환 타입: list 개수: 1
doc_id: D01
score: 0.526
content: 미르시립도서관의 정기 휴관일은 매주 월요일입니다.


<h3>5. k만 2로 바꾸어 비교</h3><p>질문·문서·임베딩 모델은 그대로 두고 반환 개수만 늘립니다.</p>

In [6]:
# hits_k2는 같은 query에서 k만 2로 변경한 결과입니다.
# 반복문으로 두 결과의 ID와 점수를 모두 보여 줍니다.
hits_k2 = store.similarity_search_with_score(query, k=2)
print("k=1 개수:", len(hits_k1))
print("k=2 개수:", len(hits_k2))
for document, score in hits_k2:
    print(document.metadata["doc_id"], round(score, 3))
assert hits_k1[0][0].metadata["doc_id"] == "D01"
assert len(hits_k2) == 2

k=1 개수: 1
k=2 개수: 2
D01 0.526
D03 0.325


<h3>정리</h3><p>임베딩 검색은 질문과 의미가 가까운 D01을 첫 결과로 반환했습니다. <code>k</code>만 늘리면 첫 결과는 유지되고 추가 후보가 보입니다.</p>

<p><strong>원인</strong> — <code>k</code>는 개수이므로 정수여야 하지만 문자열 <code>"two"</code>를 전달했습니다.</p>

In [7]:
# 문자열 k는 결과 슬라이싱에 사용할 수 없어 TypeError를 발생시킵니다.
# except는 예상한 타입 오류만 포착하고 메시지를 출력합니다.
try:
    store.similarity_search(query, k="two")
except TypeError as error:
    print("예상한 실패:", error)

예상한 실패: slice indices must be integers or None or have an __index__ method


<p><strong>수정</strong> — <code>k</code>를 정수 2로 바꾸어 동일한 질문을 다시 검색합니다.</p>

In [8]:
# fixed_hits은 k=2로 정상 실행한 Document 리스트입니다.
# 중간 결과를 출력한 뒤 개수와 D01 포함 여부를 검사합니다.
fixed_hits = store.similarity_search(query, k=2)
fixed_ids = []
for document in fixed_hits:
    fixed_ids.append(document.metadata["doc_id"])
print("After:", fixed_ids)
assert len(fixed_hits) == 2
assert "D01" in fixed_ids

After: ['D01', 'D03']
